In [2]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
from collections import Counter

In [6]:
# Ensure required NLTK data packages are available.
# nltk.data.find raises LookupError when a resource is missing, not DownloadError.
# We check the expected resource names and call nltk.download when needed.
required_nltk_resources = {
    'vader_lexicon': 'sentiment/vader_lexicon',
    'stopwords': 'corpora/stopwords'
}
for pkg_name, pkg_path in required_nltk_resources.items():
    try:
        nltk.data.find(pkg_path)
        print(f"NLTK resource '{pkg_name}' found (path: {pkg_path}).")
    except LookupError:
        print(f"NLTK resource '{pkg_name}' not found. Downloading...")
        nltk.download(pkg_name)

NLTK resource 'vader_lexicon' not found. Downloading...
NLTK resource 'stopwords' found (path: corpora/stopwords).


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\prane\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [7]:
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    print("Warning: spaCy 'en_core_web_lg' model not found. Install with 'python -m spacy download en_core_web_lg'")
    nlp = None # Fallback if spaCy model is missing

In [8]:
# Initialize the VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Define score thresholds for the four granular categories
# These thresholds are critical and will need fine-tuning based on your specific government document dataset.
SENTIMENT_THRESHOLDS = {
    "Strongly Negative": -0.6,
    "Negative": -0.1,
    "Neutral": 0.1,
    "Positive": 0.6,
    "Strongly Positive": 1.0 # Max score
}

In [9]:
def infer_topic_from_text(text: str, top_n: int = 3) -> str:
    """
    Infers the primary topic of the document using spaCy and frequency analysis of Noun Chunks.

    Args:
        text: The document text (speech, report, etc.).
        top_n: The number of top noun chunks to consider for the composite topic.

    Returns:
        A string representing the most likely topic.
    """
    if not nlp:
        return "Topic Inference Failed (spaCy model missing)"

    # Pre-process text (clean up and run through spaCy)
    doc = nlp(text)

    # 1. Extract Noun Chunks (e.g., 'climate change', 'economic stability')
    noun_chunks = [
        chunk.text.lower().strip()
        for chunk in doc.noun_chunks
        if len(chunk.text.split()) > 1 # Filter single-word noun chunks for better context
    ]

    # 2. Filter out common stopwords and irrelevant phrases
    stop_words = set(stopwords.words('english'))
    # Add common government/document boilerplate phrases to stop words
    gov_stopwords = {"government", "department", "document", "report", "official", "policy", "statement", "program", "strategy"}

    filtered_chunks = []
    for chunk in noun_chunks:
        # Simple stopword check on chunk words
        words = [word for word in chunk.split() if word not in stop_words and word not in gov_stopwords]
        if words:
            # Reconstruct the filtered chunk
            filtered_chunks.append(" ".join(words))

    # 3. Count frequency and determine the top topic
    if not filtered_chunks:
        return "General Affairs" # Default topic if no strong noun chunks are found

    chunk_counts = Counter(filtered_chunks)
    most_common = chunk_counts.most_common(top_n)

    # Combine the top N most common, weighted noun chunks into a single topic string
    topics = [item[0] for item in most_common]
    return " / ".join(topics)


In [10]:
def map_score_to_category(compound_score: float) -> str:
    """
    Maps the VADER compound sentiment score to one of the four defined categories.

    Args:
        compound_score: The single float score from VADER (-1.0 to 1.0).

    Returns:
        The corresponding sentiment category string.
    """
    if compound_score <= SENTIMENT_THRESHOLDS["Strongly Negative"]:
        return "Strongly Negative attitude towards the topic"
    elif compound_score < SENTIMENT_THRESHOLDS["Negative"]:
        return "Negative attitude towards the topic"
    elif compound_score < SENTIMENT_THRESHOLDS["Neutral"]:
        # We introduce Neutral to handle non-opinionated or balanced texts
        return "Neutral attitude towards the topic"
    elif compound_score < SENTIMENT_THRESHOLDS["Positive"]:
        return "Positive attitude towards the topic"
    else: # compound_score >= SENTIMENT_THRESHOLDS["Positive"]
        return "Strongly Positive attitude towards the topic"

def analyze_sentiment(text: str) -> dict:
    """
    The main analysis function that combines topic inference and sentiment classification.

    Args:
        text: The input document or speech.

    Returns:
        A dictionary containing the inferred topic and classified sentiment.
    """
    print("--- Starting Analysis ---")

    # 1. Infer Topic
    inferred_topic = infer_topic_from_text(text)

    # 2. Get VADER Sentiment Scores
    # Note: VADER is fast, but it is trained on social media text. For highly formal,
    # governmental language, consider replacing this with a fine-tuned BERT or RoBERTa model
    # (e.g., from Hugging Face's 'transformers' library) for significantly better accuracy
    # on nuance and context.
    vs = analyzer.polarity_scores(text)
    compound_score = vs['compound']

    # 3. Classify Sentiment
    sentiment_category = map_score_to_category(compound_score)

    print(f"Raw Compound Score: {compound_score}")
    print("--- Analysis Complete ---")

    return {
        "topic": inferred_topic,
        "sentiment_category": sentiment_category,
        "raw_compound_score": compound_score
    }
